# 09 — DOE — 2-way & 3-way interactions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-systems-today/cubespec/blob/main/notebooks/09_doe_interactions.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ai-systems-today/cubespec/main?urlpath=lab/tree/notebooks/09_doe_interactions.ipynb) [![nbviewer](https://img.shields.io/badge/render-nbviewer-orange)](https://nbviewer.org/github/ai-systems-today/cubespec/blob/main/notebooks/09_doe_interactions.ipynb) [![Dashboard](https://img.shields.io/badge/open-dashboard-2563eb)](https://sensitive-spark.lovable.app)

**Abstract.** Compute every 2-way and 3-way interaction on the full 2⁷ design and decide whether 3-way effects are worth keeping.

**Mirrors.** **DOE** tab → *interaction order* selector and the interactions panel.


In [ ]:
# cubespec-bootstrap
# Install cubespec on first run (Colab/Binder safe; no-op if already importable).
try:
    import cubespec  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'git+https://github.com/ai-systems-today/cubespec.git'])
    import cubespec  # noqa: F401


## 1. Beyond main effects

If P4_Fx and P1_dx interact, the effect of changing one depends on the level of the other. The DOE tab's *interaction order* selector exposes 1 (main only), 2 (2-way) and 3 (2-way + 3-way). We compute all three on a full 2⁷ design and look at the largest 2-way and 3-way effects on P9.


In [ ]:
from cubespec import DEFAULT_CSP, full_factorial, interactions_2way, interactions_3way
df = full_factorial(DEFAULT_CSP, levels=2)
i2 = interactions_2way(df).query("output == 'P9_compressive_strength'").sort_values('abs', ascending=False).head(8)
i3 = interactions_3way(df).query("output == 'P9_compressive_strength'").sort_values('abs', ascending=False).head(6)
import pandas as pd
print('Top 2-way interactions on P9'); print(i2.to_string(index=False))
print('\nTop 3-way interactions on P9'); print(i3.to_string(index=False))


## 2. Heat-map of all 2-way interactions


In [ ]:
import numpy as np, matplotlib.pyplot as plt
from cubespec.params import PARAM_KEYS
all2 = interactions_2way(df).query("output == 'P9_compressive_strength'")
M = np.zeros((7, 7))
for _, r in all2.iterrows():
    a, b = r.factors.split('*')
    i, j = PARAM_KEYS.index(a), PARAM_KEYS.index(b)
    M[i, j] = M[j, i] = r.effect
fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(M, cmap='RdBu_r', vmin=-abs(M).max(), vmax=abs(M).max())
ax.set_xticks(range(7)); ax.set_yticks(range(7))
ax.set_xticklabels(PARAM_KEYS, rotation=40, ha='right', fontsize=9)
ax.set_yticklabels(PARAM_KEYS, fontsize=9)
ax.set_title('2-way interactions on P9 (MPa)')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout(); plt.show()


## 3. When does 3-way matter?

If the largest 3-way effect is at least 25% of the largest 2-way, include order = 3 in your model. Otherwise stop at 2.


In [ ]:
ratio = i3['abs'].iloc[0] / i2['abs'].iloc[0]
print(f'largest 3-way / largest 2-way = {ratio:.2%} '
      f"-> {'include order = 3' if ratio > 0.25 else 'order = 2 is enough'}")


In [ ]:
# cubespec-reproducibility-footer
import platform, sys, numpy as np, scipy, pandas as pd, cubespec
print('Reproducibility')
print('  cubespec :', cubespec.__version__)
print('  python   :', sys.version.split()[0], 'on', platform.system())
print('  numpy    :', np.__version__)
print('  scipy    :', scipy.__version__)
print('  pandas   :', pd.__version__)
